# 01 Data Discovery

Common Voice Yue-only data audit for the first Cantonese ASR baseline.

In [1]:
from pathlib import Path
import os
import sys

def running_in_colab():
    return 'google.colab' in sys.modules or Path('/content').exists()

def mount_drive_if_available():
    if not running_in_colab():
        return
    try:
        from google.colab import drive
        if not Path('/content/drive').exists():
            drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {exc}')

def find_project_root():
    mount_drive_if_available()
    cwd = Path.cwd().resolve()
    env_root = os.getenv('PROJECT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path('/content/Cantonese-Speech-AI'),
        Path('/content/drive/MyDrive/Cantonese-Speech-AI'),
        Path('/content/drive/My Drive/Cantonese-Speech-AI'),
        Path(r'D:/GitHub/Cantonese-Speech-AI'),
        Path(r'D:\\GitHub\\Cantonese-Speech-AI'),
        Path('/mnt/d/GitHub/Cantonese-Speech-AI'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'cantonese_speech_ai').exists():
            return candidate.resolve()
    checked = '\n'.join(str(path) for path in candidates[:12])
    raise FileNotFoundError(
        'Cannot find the Cantonese-Speech-AI project folder in this runtime.\n\n'
        'Colab fix:\n'
        '1. Upload or copy the whole Cantonese-Speech-AI folder to Google Drive.\n'
        '2. Expected path: /content/drive/MyDrive/Cantonese-Speech-AI\n'
        '3. The folder must contain src/cantonese_speech_ai and Mozilla-HK-Speech-datasets.\n'
        '4. Then restart runtime and run this cell again.\n\n'
        'Alternative: set os.environ[\'PROJECT_ROOT\'] to your folder path before this cell.\n\n'
        f'Current cwd: {cwd}\nChecked:\n{checked}'
    )

ROOT = find_project_root()
os.environ['PROJECT_ROOT'] = str(ROOT)
default_dataset = ROOT / 'Mozilla-HK-Speech-datasets' / 'cv-corpus-26.0-2026-06-12' / 'yue'
os.environ.setdefault('COMMON_VOICE_YUE_ROOT', str(default_dataset))

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from cantonese_speech_ai.common_voice import (
    CommonVoicePaths,
    count_missing_audio,
    prepare_split,
    read_clip_durations,
    read_split,
)

paths = CommonVoicePaths.from_env()
if not paths.split_path('train').exists():
    raise FileNotFoundError(f'Dataset train.tsv not found: {paths.split_path("train")}')
ROOT, paths


(WindowsPath('D:/GitHub/Cantonese-Speech-AI'),
 CommonVoicePaths(root=WindowsPath('D:/GitHub/Cantonese-Speech-AI/Mozilla-HK-Speech-datasets/cv-corpus-26.0-2026-06-12/yue')))

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 4060 Laptop GPU


## Split counts

In [3]:
splits = ['train', 'dev', 'test', 'validated']
split_counts = {split: len(read_split(split, paths)) for split in splits}
split_counts

{'train': 7420, 'dev': 5130, 'test': 5130, 'validated': 191477}

## Prepare metadata

In [4]:
train = pd.DataFrame(prepare_split('train', paths))
dev = pd.DataFrame(prepare_split('dev', paths))
test = pd.DataFrame(prepare_split('test', paths))
durations = read_clip_durations(paths)

summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train), 'missing_audio': count_missing_audio(train.to_dict('records'))},
    {'split': 'dev', 'rows': len(dev), 'missing_audio': count_missing_audio(dev.to_dict('records'))},
    {'split': 'test', 'rows': len(test), 'missing_audio': count_missing_audio(test.to_dict('records'))},
    {'split': 'clip_durations', 'rows': len(durations), 'missing_audio': None},
])
summary

,split,rows,missing_audio
0,train,7420,0.0
1,dev,5130,0.0
2,test,5130,0.0
3,clip_durations,279366,NaN


## Text and duration statistics

In [5]:
for frame in (train, dev, test):
    frame['sentence_length'] = frame['sentence'].fillna('').str.len()

train[['duration_sec', 'sentence_length', 'up_votes', 'down_votes']].describe()

,duration_sec,sentence_length
count,7420.000000,7420.000000
mean,4.023243,10.747439
std,1.482325,5.855849
min,1.656000,2.000000
25%,3.060000,7.000000
50%,3.636000,9.000000
75%,4.608000,13.000000
max,15.228000,72.000000


## Speaker metadata

In [6]:
metadata_summary = {
    'top_accents': train['accents'].fillna('').replace('', 'unknown').value_counts().head(10),
    'age': train['age'].fillna('unknown').replace('', 'unknown').value_counts(),
    'gender': train['gender'].fillna('unknown').replace('', 'unknown').value_counts(),
}
metadata_summary

{'top_accents': accents
 unknown                      4286
 廣州音                           967
 广州音                           955
 佛山音                           284
 香港                            190
 广州                            184
 90後土生土長澳門人                    133
 Hong Kong                     103
 廣西白話|邕潯白話|桂平白話|南寧白話            75
 Canton|Gwongzau|Guangzhou      57
 Name: count, dtype: int64,
 'age': age
 twenties    2333
 unknown     1955
 thirties    1830
 fourties    1068
 teens        107
 fifties      107
 sixties       20
 Name: count, dtype: int64,
 'gender': gender
 female_feminine    3878
 unknown            2719
 male_masculine      823
 Name: count, dtype: int64}

## UTF-8 sample check

In [7]:
train[['audio_path', 'sentence', 'normalized_sentence', 'accents', 'duration_sec']].head(10)

,audio_path,sentence,normalized_sentence,accents,duration_sec
0,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,大陸有專俾同志用嘅交友程式？,大陸有專俾同志用嘅交友程式,广东音,4.608
1,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,仲有舊年年尾話唔應該再猛咁撩鼻，不如做抗體檢測好過，個訪問冇耐就喺大陸全網禁咗,仲有舊年年尾話唔應該再猛咁撩鼻 不如做抗體檢測好過 個訪問冇耐就喺大陸全網禁咗,广州音,10.080
2,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,邊有噉大隻蛤乸隨街跳,邊有噉大隻蛤乸隨街跳,广州音,3.528
3,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,但係我唔特登講就成日會俾人估錯隔離,但係我唔特登講就成日會俾人估錯隔離,广州音,5.148
4,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,除非有人喺第二區截車入去,除非有人喺第二區截車入去,广州音,3.636
5,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,注定有緣冇瞓,注定有緣冇瞓,广州音,2.448
6,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,雖然我明點解驚，我睇到個個冇口罩見晒樣都心肝卜卜跳,雖然我明點解驚 我睇到個個冇口罩見晒樣都心肝卜卜跳,广州音,7.848
7,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,本身前面啲陰我都想成束剪起十五厘米，髮型師話留嚟遮返後面剷咗嘅位唔剪到咁盡,本身前面啲陰我都想成束剪起十五厘米 髮型師話留嚟遮返後面剷咗嘅位唔剪到咁盡,广州音,10.908
8,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,同埋話女仔唔好咁易俾條仔食，有咩留返結婚先，拍拖錫下攬下無傷大雅，其他就唔好嘞,同埋話女仔唔好咁易俾條仔食 有咩留返結婚先 拍拖錫下攬下無傷大雅 其他就唔好嘞,广州音,10.728
9,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,我聽到去屢敗屢戰個到就冇聽,我聽到去屢敗屢戰個到就冇聽,广州音,4.176
